# Triage Module Evaluation

This notebook evaluates the `TriageModule` (Linguistic Normalizer and Language Detector) using a custom dataset. It fills in the `Detected Language` and `Normalized Output` columns for each entry in the evaluation dataset.

In [1]:
# 0. Install required libraries
%pip install pandas openpyxl tqdm requests python-dotenv rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 242.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 175.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 13.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 2.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 146.3 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import time
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# 1. Add project root to sys.path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.config import FrameworkConfig

# 2. Load environment variables from project root .env
load_dotenv(os.path.join(project_root, '.env'))

print("Environment setup complete.")

Environment setup complete.


/home/perhapsyou/Desktop/404FoundUs/Legal-Adaptive-Routing-Framework/myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ------------------------------------------------------------------
# CONFIGURATION: Manually paste your OpenRouter API key here
# ------------------------------------------------------------------
OPENROUTER_API_KEY = "" # <--- PASTE API KEY HERE
# ------------------------------------------------------------------

if not OPENROUTER_API_KEY:
    print("No manual API key provided. Falling back to .env or environment variables.")
    api_key = os.getenv("OPENROUTER_API_KEY")
else:
    print("Using manually provided API key.")
    api_key = OPENROUTER_API_KEY

if not api_key:
    print("WARNING: No API key found. Module initialization might fail.")

Using manually provided API key.


In [4]:
# Initialize Triage Module
triage = TriageModule(api_key=api_key)

print(f"Triage Module initialized using model: {FrameworkConfig._TRIAGE_MODEL}")

Triage Module initialized using model: qwen/qwen-turbo


In [5]:
# Path to the evaluation dataset
dataset_path = 'dataset/Triage-Module-Evaluation-Dataset.xlsx'

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

# Load the dataset
df = pd.read_excel(dataset_path)

print(f"Loaded dataset with {len(df)} rows.")
print("Columns:", df.columns.tolist())

# Identify the input column (change this matches your Excel file)
input_column = "Prompt" 
if input_column not in df.columns:
    # Attempting to find common names if 'Prompt' is not present
    candidates = ["Input", "Query", "Prompt", "Text", "raw_text", "prompts"]
    for cand in candidates:
        if cand in df.columns:
            input_column = cand
            print(f"Using '{input_column}' as the input column.")
            break

df.head()

Loaded dataset with 149 rows.
Columns: ['prompts', 'language', 'llm_normalized_output', 'llm_detected_language']
Using 'prompts' as the input column.


,prompts,language,llm_normalized_output,llm_detected_language
0,"Attorney, hawak po ng employer ko ang passport...",Tagalog,NaN,NaN
1,Hindi ako pinayagan mag-day off ng amo ko for ...,Tagalog,NaN,NaN
2,May karapatan ba akong magreklamo kapag sinasa...,Tagalog,NaN,NaN
3,Gusto ko na po umuwi pero ayaw ibigay ang tick...,Tagalog,NaN,NaN
4,Pinapirma ako ng kontrata na iba sa pinag-usap...,Tagalog,NaN,NaN


In [6]:
# Configuration for robustness
lang_col = "Detected Language"
norm_col = "Normalized Output"
max_retries = 10
retry_delay = 5  # seconds
autosave_interval = 5  # Save every 5 successful requests
checkpoint_path = 'dataset/Triage-Module-Evaluation-Checkpoint.xlsx'

# Initialize columns if they don't exist
if lang_col not in df.columns:
    df[lang_col] = None
if norm_col not in df.columns:
    df[norm_col] = None

print("Starting evaluation processing...")

success_count = 0
for index, row in tqdm(df.iterrows(), total=len(df)):
    # Skip if already processed (helps with resuming from checkpoint)
    if pd.notna(row[lang_col]) and row[lang_col] != "ERROR":
        continue

    raw_input = row[input_column]
    if pd.isna(raw_input) or str(raw_input).strip() == "":
        continue

    attempt = 0
    while attempt < max_retries:
        try:
            # Execute triage processing
            result = triage._process_request_(str(raw_input))
            
            # Update the dataframe
            df.at[index, lang_col] = result.get("detected_language")
            df.at[index, norm_col] = result.get("normalized_text")
            
            success_count += 1
            
            # Periodic autosave to checkpoint
            if success_count % autosave_interval == 0:
                df.to_excel(checkpoint_path, index=False)
            
            break  # Success, exit retry loop
            
        except Exception as e:
            attempt += 1
            if attempt < max_retries:
                print(f"Error on row {index} (Attempt {attempt}/{max_retries}): {e}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                print(f"Failed row {index} after {max_retries} attempts. Final error: {e}")
                df.at[index, lang_col] = "ERROR"
                df.at[index, norm_col] = str(e)

print("Processing complete.")

Starting evaluation processing...


100%|██████████| 149/149 [02:14<00:00,  1.11it/s]

Processing complete.


In [7]:
# Save the final results
output_path = 'dataset/Triage-Module-Evaluation-Results.xlsx'
df.to_excel(output_path, index=False)

print(f"Final results saved to: {output_path}")
# Remove checkpoint if finished successfully (optional, commented out for safety)
# if os.path.exists(checkpoint_path): os.remove(checkpoint_path)
df.head()

Final results saved to: dataset/Triage-Module-Evaluation-Results.xlsx


,prompts,language,llm_normalized_output,llm_detected_language,Detected Language,Normalized Output
0,"Attorney, hawak po ng employer ko ang passport...",Tagalog,NaN,NaN,Tagalog,The employer holds the passport of the employe...
1,Hindi ako pinayagan mag-day off ng amo ko for ...,Tagalog,NaN,NaN,Tagalog,I was not permitted to take a 3-month leave by...
2,May karapatan ba akong magreklamo kapag sinasa...,Tagalog,NaN,NaN,Tagalog,Can I file a complaint if my pet is harming me?
3,Gusto ko na po umuwi pero ayaw ibigay ang tick...,Tagalog,NaN,NaN,Tagalog,I wish to return home but the ticket is not be...
4,Pinapirma ako ng kontrata na iba sa pinag-usap...,Tagalog,NaN,NaN,Tagalog,I was presented with a contract different from...
